# 📊 Módulo 6 — Reportes y Dashboard Gerencial
**Sistema de Gestión de Recursos Humanos**

Este módulo genera los indicadores clave (KPIs) y el dashboard gerencial:
- **Plantilla activa** — headcount por departamento
- **Rotación** — tasa mensual y acumulada
- **Asistencia / Ausencias** — porcentaje por empleado y global
- **Vacaciones** — días tomados vs. pendientes
- **Capacitaciones** — completadas y en curso
- **Desempeño** — ranking y promedio por departamento
- **Exportar CSV** — cualquier tabla a archivo

> **Base de datos:** `rrhh_sistema` · **Puerto:** `1989` · **Motor:** MySQL


In [ ]:
# ── 1. Instalación automática de dependencias ─────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'mysql-connector-python', 'ipywidgets', 'pandas',
                'matplotlib', '--quiet'], check=False)

import mysql.connector
from mysql.connector import Error
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from datetime import date
import io, base64

matplotlib.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print('✅ Librerías cargadas correctamente.')


In [ ]:
# Conexion a la base de datos
import os
import mysql.connector

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': int(os.getenv('DB_PORT', '1989')),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'rrhh_sistema'),
    'charset': 'utf8mb4',
}

def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

try:
    conn = get_conn()
    conn.close()
    print(f'Conexion exitosa -> {DB_CONFIG["database"]} (Puerto {DB_CONFIG["port"]})')
except mysql.connector.Error as e:
    print(f'Error de conexion: {e}')



In [ ]:
# ── 3. Funciones de reportes ──────────────────────────────────────────────

def rpt_plantilla_activa() -> pd.DataFrame:
    conn = get_conn()
    df = pd.read_sql("""
        SELECT d.nombre_departamento AS Departamento,
               COUNT(*) AS Empleados_Activos
        FROM   empleados e
        JOIN   personal      p ON p.codigo_empresa  = e.codigo_empresa
        JOIN   departamentos d ON d.id_departamento = p.id_departamento
        WHERE  e.estado = 'activo'
        GROUP  BY d.nombre_departamento
        ORDER  BY Empleados_Activos DESC
    """, conn); conn.close()
    return df

def rpt_rotacion() -> pd.DataFrame:
    conn = get_conn()
    df = pd.read_sql("""
        SELECT
            DATE_FORMAT(sp.fecha_salida, '%Y-%m') AS Mes,
            COUNT(*)                              AS Salidas,
            ROUND(COUNT(*) * 100.0 /
                  NULLIF((SELECT COUNT(*) FROM empleados WHERE estado='activo'),0),2
                 )                                AS Tasa_Rotacion_Pct
        FROM salida_personal sp
        GROUP BY Mes
        ORDER BY Mes DESC
        LIMIT 12
    """, conn); conn.close()
    return df

def rpt_asistencia() -> pd.DataFrame:
    conn = get_conn()
    df = pd.read_sql("""
        SELECT CONCAT(e.nombre,' ',e.apellido) AS Empleado,
               d.nombre_departamento            AS Departamento,
               SUM(a.presente)                  AS Dias_Presentes,
               SUM(1 - a.presente)              AS Dias_Falta,
               COUNT(*)                         AS Total_Registros,
               ROUND(SUM(a.presente)*100.0/NULLIF(COUNT(*),0),1) AS Pct_Asistencia
        FROM   asistencias a
        JOIN   empleados    e ON e.codigo_empresa  = a.codigo_empresa
        JOIN   personal     p ON p.codigo_empresa  = e.codigo_empresa
        JOIN   departamentos d ON d.id_departamento = p.id_departamento
        WHERE  MONTH(a.fecha) = MONTH(CURDATE())
          AND  YEAR(a.fecha)  = YEAR(CURDATE())
        GROUP  BY e.codigo_empresa
        ORDER  BY Pct_Asistencia
    """, conn); conn.close()
    return df

def rpt_ausencias() -> pd.DataFrame:
    conn = get_conn()
    df = pd.read_sql("""
        SELECT CONCAT(e.nombre,' ',e.apellido) AS Empleado,
               d.nombre_departamento           AS Departamento,
               COUNT(au.id_ausencia)           AS Total_Ausencias,
               GROUP_CONCAT(DISTINCT au.motivo ORDER BY au.fecha SEPARATOR ' | ') AS Motivos
        FROM   ausencias     au
        JOIN   empleados     e  ON e.codigo_empresa  = au.codigo_empresa
        JOIN   personal      p  ON p.codigo_empresa  = e.codigo_empresa
        JOIN   departamentos d  ON d.id_departamento = p.id_departamento
        WHERE  MONTH(au.fecha) = MONTH(CURDATE())
          AND  YEAR(au.fecha)  = YEAR(CURDATE())
        GROUP  BY e.codigo_empresa
        ORDER  BY Total_Ausencias DESC
    """, conn); conn.close()
    return df

def rpt_vacaciones() -> pd.DataFrame:
    conn = get_conn()
    df = pd.read_sql("""
        SELECT CONCAT(e.nombre,' ',e.apellido) AS Empleado,
               d.nombre_departamento           AS Departamento,
               SUM(v.dias_tomados)             AS Dias_Vacaciones_Tomados
        FROM   vacaciones    v
        JOIN   empleados     e ON e.codigo_empresa  = v.codigo_empresa
        JOIN   personal      p ON p.codigo_empresa  = e.codigo_empresa
        JOIN   departamentos d ON d.id_departamento = p.id_departamento
        GROUP  BY e.codigo_empresa
        ORDER  BY Dias_Vacaciones_Tomados DESC
    """, conn); conn.close()
    return df

def rpt_capacitaciones() -> pd.DataFrame:
    conn = get_conn()
    df = pd.read_sql("""
        SELECT CONCAT(e.nombre,' ',e.apellido) AS Empleado,
               d.nombre_departamento           AS Departamento,
               COUNT(c.id_capacitacion)        AS Total_Capacitaciones,
               SUM(c.fecha_fin IS NOT NULL AND c.fecha_fin <= CURDATE()) AS Completadas,
               SUM(c.fecha_fin IS NULL  OR  c.fecha_fin  > CURDATE())   AS En_Curso
        FROM   capacitaciones c
        JOIN   empleados      e ON e.codigo_empresa  = c.codigo_empresa
        JOIN   personal       p ON p.codigo_empresa  = e.codigo_empresa
        JOIN   departamentos  d ON d.id_departamento = p.id_departamento
        GROUP  BY e.codigo_empresa
        ORDER  BY Total_Capacitaciones DESC
    """, conn); conn.close()
    return df

def rpt_desempeno() -> pd.DataFrame:
    conn = get_conn()
    df = pd.read_sql("""
        SELECT CONCAT(e.nombre,' ',e.apellido)       AS Empleado,
               d.nombre_departamento                 AS Departamento,
               ROUND(e.pct_desempeno, 2)             AS Promedio_Desempeno_Neto,
               COUNT(ev.id_evaluacion)               AS Total_Evaluaciones
        FROM   empleados            e
        JOIN   personal             p  ON p.codigo_empresa  = e.codigo_empresa
        JOIN   departamentos        d  ON d.id_departamento = p.id_departamento
        LEFT   JOIN evaluaciones_desempeno ev ON ev.codigo_empresa = e.codigo_empresa
        WHERE  e.estado = 'activo'
        GROUP  BY e.codigo_empresa
        ORDER  BY Promedio_Desempeno_Neto DESC
    """, conn); conn.close()
    return df

def df_to_csv_link(df: pd.DataFrame, nombre: str) -> str:
    csv_bytes  = df.to_csv(index=False).encode('utf-8-sig')
    b64        = base64.b64encode(csv_bytes).decode()
    return (f"<a href='data:text/csv;base64,{b64}' download='{nombre}.csv' "
            f"style='font-size:13px'>⬇️ Descargar {nombre}.csv</a>")

print('✅ Funciones de reportes cargadas.')


In [ ]:
# ── 4. KPIs rápidos ───────────────────────────────────────────────────────

def mostrar_kpis():
    try:
        conn = get_conn(); cur = conn.cursor()

        cur.execute("SELECT COUNT(*) FROM empleados WHERE estado='activo'")
        total_activos = cur.fetchone()[0]

        cur.execute("""SELECT COUNT(*) FROM salida_personal
                        WHERE MONTH(fecha_salida)=MONTH(CURDATE())
                          AND YEAR(fecha_salida)=YEAR(CURDATE())""")
        salidas_mes = cur.fetchone()[0]

        cur.execute("""SELECT ROUND(AVG(presente)*100,1) FROM asistencias
                        WHERE MONTH(fecha)=MONTH(CURDATE()) AND YEAR(fecha)=YEAR(CURDATE())""")
        pct_asistencia = cur.fetchone()[0] or 0

        cur.execute("SELECT ROUND(AVG(pct_desempeno),1) FROM empleados WHERE pct_desempeno IS NOT NULL")
        avg_desempeno = cur.fetchone()[0] or 'N/D'

        cur.execute("""SELECT COUNT(*) FROM capacitaciones
                        WHERE fecha_fin IS NULL OR fecha_fin > CURDATE()""")
        caps_activas = cur.fetchone()[0]

        cur.close(); conn.close()

        tasa_rot = round(salidas_mes * 100 / total_activos, 2) if total_activos else 0

        kpi_html = f"""
        <div style='display:flex;flex-wrap:wrap;gap:12px;margin:10px 0'>
          <div style='background:#e3f2fd;border-radius:10px;padding:16px 24px;min-width:140px;text-align:center'>
            <div style='font-size:28px;font-weight:bold;color:#1565c0'>{total_activos}</div>
            <div style='color:#555;margin-top:4px'>👥 Empleados Activos</div>
          </div>
          <div style='background:#fce4ec;border-radius:10px;padding:16px 24px;min-width:140px;text-align:center'>
            <div style='font-size:28px;font-weight:bold;color:#c62828'>{tasa_rot}%</div>
            <div style='color:#555;margin-top:4px'>🔄 Rotación (mes)</div>
          </div>
          <div style='background:#e8f5e9;border-radius:10px;padding:16px 24px;min-width:140px;text-align:center'>
            <div style='font-size:28px;font-weight:bold;color:#2e7d32'>{pct_asistencia}%</div>
            <div style='color:#555;margin-top:4px'>📌 Asistencia (mes)</div>
          </div>
          <div style='background:#ede7f6;border-radius:10px;padding:16px 24px;min-width:140px;text-align:center'>
            <div style='font-size:28px;font-weight:bold;color:#4527a0'>{avg_desempeno}</div>
            <div style='color:#555;margin-top:4px'>📊 Desempeño Prom.</div>
          </div>
          <div style='background:#fff8e1;border-radius:10px;padding:16px 24px;min-width:140px;text-align:center'>
            <div style='font-size:28px;font-weight:bold;color:#e65100'>{caps_activas}</div>
            <div style='color:#555;margin-top:4px'>📚 Caps. en Curso</div>
          </div>
        </div>
        """
        display(HTML(f"<h3 style='color:#1a73e8'>📈 KPIs Generales — {date.today().strftime('%B %Y')}</h3>"))
        display(HTML(kpi_html))
    except Exception as ex:
        print(f'❌ Error al calcular KPIs: {ex}')

mostrar_kpis()


In [ ]:
# ── 5. PANEL DE REPORTES CON PESTAÑAS ────────────────────────────────────

BTN_LY = widgets.Layout(width='220px', margin='6px 4px 0 0')
OUT_LY = widgets.Layout(margin='8px 0 0 0', border='1px solid #e0e0e0',
                         padding='10px', border_radius='6px', min_height='50px')

def make_tab(titulo, emoji, btn_label, query_fn, chart_fn=None):
    btn_data  = widgets.Button(description=f'🔍 {btn_label}',
                                button_style='primary', layout=BTN_LY)
    btn_csv   = widgets.Button(description='⬇️ Exportar CSV',
                                button_style='info',    layout=BTN_LY)
    out_tabla = widgets.Output(layout=OUT_LY)
    out_chart = widgets.Output(layout=OUT_LY)
    _cache    = {}

    def on_data(b):
        with out_tabla:
            clear_output()
            try:
                df = query_fn(); _cache['df'] = df
                if df.empty: print('Sin datos disponibles.')
                else:
                    display(HTML(f"<b>{emoji} {titulo}</b>"))
                    display(df.style.set_table_styles([
                        {'selector':'th','props':[('background','#1a73e8'),
                                                  ('color','white'),('padding','5px 10px')]},
                        {'selector':'td','props':[('padding','4px 10px')]}
                    ]).hide(axis='index'))
                    if chart_fn:
                        with out_chart: clear_output(); chart_fn(df)
            except Exception as ex:
                print(f'❌ Error: {ex}')

    def on_csv(b):
        with out_tabla:
            if 'df' not in _cache: print('Primero carga los datos.')
            else: display(HTML(df_to_csv_link(_cache['df'], titulo.replace(' ','_'))))

    btn_data.on_click(on_data); btn_csv.on_click(on_csv)
    return widgets.VBox([
        widgets.HTML(f"<h3 style='color:#1a73e8'>{emoji} {titulo}</h3>"),
        widgets.HBox([btn_data, btn_csv]),
        out_tabla, out_chart
    ], layout=widgets.Layout(padding='14px'))

# ── Gráficas ──────────────────────────────────────────────────────────────
def chart_plantilla(df):
    if df.empty: return
    fig, ax = plt.subplots(figsize=(7,3.5))
    ax.barh(df['Departamento'], df['Empleados_Activos'], color='#1a73e8')
    ax.set_xlabel('Empleados'); ax.set_title('Plantilla activa por departamento')
    plt.tight_layout(); plt.show()

def chart_rotacion(df):
    if df.empty: return
    fig, ax = plt.subplots(figsize=(7,3.5))
    ax.plot(df['Mes'], df['Tasa_Rotacion_Pct'], marker='o', color='#e53935')
    ax.set_ylabel('%'); ax.set_title('Tasa de rotación mensual')
    plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

def chart_desempeno(df):
    df2 = df.dropna(subset=['Promedio_Desempeno_Neto'])
    if df2.empty: return
    fig, ax = plt.subplots(figsize=(7,4))
    colores = ['#43a047' if v >= 70 else '#ffa000' if v >= 50 else '#e53935'
               for v in df2['Promedio_Desempeno_Neto']]
    ax.barh(df2['Empleado'], df2['Promedio_Desempeno_Neto'], color=colores)
    ax.set_xlabel('Puntaje neto'); ax.set_title('Ranking de desempeño')
    ax.axvline(70, color='green', linestyle='--', linewidth=0.8, label='Meta 70')
    ax.legend(); plt.tight_layout(); plt.show()

# ── Construir pestañas ────────────────────────────────────────────────────
tabs = widgets.Tab(children=[
    make_tab('Plantilla Activa',  '👥', 'Cargar',  rpt_plantilla_activa, chart_plantilla),
    make_tab('Rotación',          '🔄', 'Cargar',  rpt_rotacion,         chart_rotacion),
    make_tab('Asistencia',        '📌', 'Cargar',  rpt_asistencia),
    make_tab('Ausencias',         '🟡', 'Cargar',  rpt_ausencias),
    make_tab('Vacaciones',        '🌴', 'Cargar',  rpt_vacaciones),
    make_tab('Capacitaciones',    '📚', 'Cargar',  rpt_capacitaciones),
    make_tab('Desempeño',         '📊', 'Cargar',  rpt_desempeno,        chart_desempeno),
])
for i, titulo in enumerate(['👥 Plantilla','🔄 Rotación','📌 Asistencia',
                              '🟡 Ausencias','🌴 Vacaciones','📚 Capacit.','📊 Desempeño']):
    tabs.set_title(i, titulo)

display(tabs)
